<a href="https://colab.research.google.com/github/James5401/Anti-Littering-System-Computer-Vision/blob/main/Sanitation%20Enforcement%20System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import os

print('Please upload your littering video file (e.g., .mp4, .mov):')
uploaded = files.upload()

if uploaded:
    fn = list(uploaded.keys())[0]
    # Rename to output.mp4 for consistency with downstream cells
    if os.path.exists('output.mp4'):
        os.remove('output.mp4')
    os.rename(fn, 'output.mp4')
    video_path = os.path.abspath('output.mp4')
    print(f'Video saved and renamed to: {video_path}')
else:
    print('Upload cancelled or failed.')

Please upload your littering video file (e.g., .mp4, .mov):


In [ ]:
!pip install -q ultralytics
import cv2
import torch
import numpy as np
import tensorflow as tf
from ultralytics import RTDETR
from google.colab import files
import os
import time

def run_inference(interpreter, input_size, frame):
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (input_size, input_size))
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    if input_details[0]['dtype'] == np.uint8:
        input_image = np.expand_dims(image, axis=0).astype(np.uint8)
    else:
        input_image = np.expand_dims(image, axis=0).astype(np.float32) / 255.0
    interpreter.set_tensor(input_details[0]['index'], input_image)
    interpreter.invoke()
    keypoints = interpreter.get_tensor(output_details[0]['index'])
    h, w, _ = frame.shape
    keypoints = keypoints[0, 0, :, :]
    keypoints[:, 0] *= h
    keypoints[:, 1] *= w
    return keypoints[:, [1, 0]].astype(int), keypoints[:, 2]

# Ensure model directory exists
if not os.path.exists('MoveNet/singlepose_thunder_3.tflite'):
    print('Downloading MoveNet model...')
    !mkdir -p MoveNet
    !wget -q -O MoveNet/singlepose_thunder_3.tflite https://tfhub.dev/google/lite-model/movenet/singlepose/thunder/tflite/float16/4?lite-format=tflite

interpreter = tf.lite.Interpreter(model_path='MoveNet/singlepose_thunder_3.tflite')
interpreter.allocate_tensors()
detector = RTDETR('rtdetr-l.pt')

video_path = '/content/output.mp4'
if not os.path.exists(video_path):
    print('Error: output.mp4 not found. Please upload a video first.')
else:
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width, frame_height = int(cap.get(3)), int(cap.get(4))
    fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30
    out = cv2.VideoWriter('processed_litter_labeled.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))
    object_hand_history = {}

    print(f'Starting processing: {total_frames} frames total...')
    start_time = time.time()
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        results = detector.track(frame, persist=True, tracker='botsort.yaml', conf=0.05, imgsz=640, verbose=False)
        keypoints, scores = run_inference(interpreter, 256, frame)
        annotated_frame = frame.copy()
        wrists = [9, 10]
        litter_alert_active = False

        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy()
            clss = results[0].boxes.cls.cpu().numpy()

            for box, track_id, cls in zip(boxes, ids, clss):
                obj_center = ((box[0]+box[2])/2, (box[1]+box[3])/2)
                tid = int(track_id)
                label_name = detector.names[int(cls)]
                min_dist = 9999
                for w_idx in wrists:
                    if scores[w_idx] > 0.2:
                        d = np.linalg.norm(np.array(obj_center) - np.array(keypoints[w_idx]))
                        if d < min_dist: min_dist = d

                is_held = min_dist < 180
                if tid not in object_hand_history: object_hand_history[tid] = {'was_held': False}
                if is_held: object_hand_history[tid]['was_held'] = True

                if object_hand_history[tid]['was_held'] and not is_held and obj_center[1] > (frame_height * 0.5):
                    litter_alert_active = True
                    color = (0, 0, 255)
                else:
                    color = (0, 255, 255) if is_held else (0, 255, 0)

                cv2.rectangle(annotated_frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), color, 3)
                cv2.putText(annotated_frame, f'{label_name} ID:{tid}', (int(box[0]), int(box[1]-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        if litter_alert_active:
            cv2.rectangle(annotated_frame, (0, 0), (frame_width, 100), (0, 0, 255), -1)
            cv2.putText(annotated_frame, 'LITTERING DETECTED!', (int(frame_width/4), 70), cv2.FONT_HERSHEY_DUPLEX, 2, (255, 255, 255), 4)

        out.write(annotated_frame)

        if frame_idx % 30 == 0:
            curr_time = time.time() - start_time
            print(f'Frame {frame_idx}/{total_frames} processed. Elapsed: {curr_time:.2f}s')
        frame_idx += 1

    cap.release()
    out.release()
    total_time = time.time() - start_time
    print(f'Processing Complete! Total time: {total_time:.2f}s. Initializing download...')
    files.download('processed_litter_labeled.mp4')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Starting processing: 80 frames total...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 254ms
Prepared 1 package in 102ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Frame 0/80 processed. Elapsed: 4.58s
Frame 30/80 processed. Elapsed: 71.75s
Frame 60/80 processed. Elapsed: 138.69s
Processing Complete! Total time: 179.96s. Initializing download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import cv2
import tensorflow as tf
import numpy as np
from ultralytics import RTDETR
import os
import time

# Optimization 1: Use RT-DETR-L
detector = RTDETR('rtdetr-l.pt')

# MoveNet setup
interpreter = tf.lite.Interpreter(model_path='MoveNet/singlepose_thunder_3.tflite')
interpreter.allocate_tensors()
input_size = 256

def run_inference_local(interpreter, input_size, frame):
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (input_size, input_size))
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Check if model expects UINT8 (0-255) or FLOAT32 (0.0-1.0)
    if input_details[0]['dtype'] == np.uint8:
        input_image = np.expand_dims(image, axis=0).astype(np.uint8)
    else:
        input_image = np.expand_dims(image, axis=0).astype(np.float32) / 255.0

    interpreter.set_tensor(input_details[0]['index'], input_image)
    interpreter.invoke()
    keypoints = interpreter.get_tensor(output_details[0]['index'])
    h, w, _ = frame.shape
    keypoints = keypoints[0, 0, :, :]
    keypoints[:, 0] *= h
    keypoints[:, 1] *= w
    return keypoints[:, [1, 0]].astype(int), keypoints[:, 2]

video_path = 'output.mp4'
cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30
out = cv2.VideoWriter('processed_optimized.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

print(f'Starting Optimized Processing: {total_frames} frames total.')
start_time = time.time()

frame_count = 0
# Initialize placeholders for tracked results between skipped frames
results = None
keypoints = None
scores = None

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # Optimization: Process inference every 2nd frame
    if frame_count % 2 == 0 or results is None:
        results = detector.track(frame, persist=True, tracker='botsort.yaml', conf=0.1, imgsz=320, verbose=False)
        keypoints, scores = run_inference_local(interpreter, input_size, frame)

    annotated_frame = frame.copy()
    wrists = [9, 10]
    near_hand_count = 0
    det_count = 0

    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy()
        det_count = len(boxes)

        for box, track_id in zip(boxes, ids):
            obj_center = ((box[0]+box[2])/2, (box[1]+box[3])/2)
            is_near_hand = False
            for w_idx in wrists:
                if scores[w_idx] > 0.2:
                    if np.linalg.norm(np.array(obj_center) - np.array(keypoints[w_idx])) < 150:
                        is_near_hand = True
                        near_hand_count += 1

            color = (0, 255, 255) if is_near_hand else (0, 165, 255)
            cv2.rectangle(annotated_frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), color, 2)

    for i in wrists:
        if scores[i] > 0.2: cv2.circle(annotated_frame, (keypoints[i][0], keypoints[i][1]), 10, (0, 255, 0), -1)

    out.write(annotated_frame)

    if frame_count % 30 == 0:
        elapsed = time.time() - start_time
        print(f'Frame {frame_count}/{total_frames} | Detections: {det_count} | Held: {near_hand_count} | Elapsed: {elapsed:.1f}s')

    frame_count += 1

cap.release()
out.release()
print(f'Finished! Processed {frame_count} frames in {time.time()-start_time:.1f}s.')

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Starting Optimized Processing: 80 frames total.
Frame 0/80 | Detections: 3 | Held: 3 | Elapsed: 1.4s
Frame 30/80 | Detections: 1 | Held: 0 | Elapsed: 19.6s
Frame 60/80 | Detections: 4 | Held: 2 | Elapsed: 32.2s
Finished! Processed 80 frames in 39.8s.


In [ ]:
!find . -name "*.tflite"

./MoveNet/singlepose_thunder_3.tflite


In [ ]:
!find . -name "*.tflite"

./MoveNet/singlepose_thunder_3.tflite


In [ ]:
import os
import requests
import sys

# 1. Clone Repository for IntegratedArchitecture
# Using env variable to prevent git from asking for credentials
repo_url = 'https://github.com/roboflow/Anti-Littering-System-Computer-Vision.git'
repo_dir = '/content/Anti-Littering-System-Computer-Vision'

if not os.path.exists(repo_dir):
    print('Cloning repository...')
    !GIT_TERMINAL_PROMPT=0 git clone {repo_url} || echo 'Clone failed, but checking for local files...'

if os.path.exists(repo_dir):
    sys.path.append(repo_dir)

# 2. Download MoveNet Model
model_dir = 'MoveNet'
model_path = os.path.join(model_dir, 'singlepose_thunder_3.tflite')
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

if not os.path.exists(model_path):
    print('Downloading MoveNet model...')
    url = 'https://tfhub.dev/google/lite-model/movenet/singlepose/thunder/tflite/float16/4?lite-format=tflite'
    response = requests.get(url)
    with open(model_path, 'wb') as f:
        f.write(response.content)
    print('MoveNet Download complete.')

# 3. Download RT-DETR weights
if not os.path.exists('rtdetr-l.pt'):
    print('Downloading RT-DETR weights...')
    !wget -q https://github.com/ultralytics/assets/releases/download/v8.2.0/rtdetr-l.pt

!pip install -q ultralytics
print('Environment Setup Complete!')

Cloning repository...
Cloning into 'Anti-Littering-System-Computer-Vision'...
fatal: could not read Username for 'https://github.com': terminal prompts disabled
Clone failed, but checking for local files...
Environment Setup Complete!


In [ ]:
!ls -R MoveNet/

MoveNet/:
singlepose_thunder_3.tflite


In [ ]:
!ls -R MoveNet/

MoveNet/:
singlepose_thunder_3.tflite


In [ ]:
import cv2
import tensorflow as tf
import numpy as np
from IntegratedArchitecture import detect_objects, run_inference
from ultralytics import YOLO

# Initialize Models
model = YOLO('best.pt')
interpreter = tf.lite.Interpreter(model_path='MoveNet/singlepose_thunder_3.tflite')
interpreter.allocate_tensors()
input_size = 256

video_path = 'output.mp4'
cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS)) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30
out = cv2.VideoWriter('processed_debug.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

print(f"Processing with Hand-Object Proximity Logic...")
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # 1. Pose Detection (MoveNet) to find hands
    keypoints, scores = run_inference(interpreter, input_size, frame)

    # 2. Object Detection (extremely sensitive)
    results = model(frame, verbose=False, conf=0.02, imgsz=640)

    annotated_frame = frame.copy()
    wrists = [9, 10] # Left and Right Wrist indices

    for r in results:
        for box in r.boxes:
            b = box.xyxy[0].cpu().numpy() # [x1, y1, x2, y2]
            obj_center = ((b[0]+b[2])/2, (b[1]+b[3])/2)

            # Check if detection is near any wrist
            is_near_hand = False
            for w_idx in wrists:
                if scores[w_idx] > 0.2:
                    dist = np.linalg.norm(np.array(obj_center) - np.array(keypoints[w_idx]))
                    if dist < 150: # Threshold for proximity
                        is_near_hand = True

            # Highlight detection
            color = (0, 255, 255) if is_near_hand else (255, 0, 0)
            cv2.rectangle(annotated_frame, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), color, 3)
            label = f"{model.names[int(box.cls)]} {'(HELD)' if is_near_hand else ''}"
            cv2.putText(annotated_frame, label, (int(b[0]), int(b[1])-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    # Draw Wrists
    for i in wrists:
        w_color = (0, 255, 0) if scores[i] > 0.2 else (0, 0, 255)
        cv2.circle(annotated_frame, (keypoints[i][0], keypoints[i][1]), 12, w_color, -1)

    out.write(annotated_frame)
    frame_count += 1
    if frame_count % 30 == 0: print(f"Processed {frame_count} frames...")

cap.release()
out.release()
print("Done! Check 'processed_debug.mp4' to see if detections are now flagged near hands.")

ModuleNotFoundError: No module named 'IntegratedArchitecture'

In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO('best.pt')
video_path = 'output.mp4'
cap = cv2.VideoCapture(video_path)

# Check a few frames throughout the video
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Checking model against {total_frames} frames...")

for frame_idx in [total_frames//4, total_frames//2, 3*total_frames//4]:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if ret:
        # Increasing imgz to 640 (standard) to improve detection
        results = model.predict(frame, conf=0.01, imgsz=640, verbose=False)
        for r in results:
            if len(r.boxes) > 0:
                print(f"--- Detections at frame {frame_idx} ---")
                for box in r.boxes:
                    print(f"Class: {model.names[int(box.cls)]}, Conf: {box.conf.item():.4f}")
            else:
                print(f"Frame {frame_idx}: No detections.")

cap.release()

In [ ]:
from google.colab import files
files.download('processed_debug.mp4')

In [ ]:
!pip install filterpy lap

In [ ]:
with open('IntegratedArchitecture.py', 'r') as f:
    print(f.read())

In [ ]:
!wget https://github.com/ultralytics/assets/releases/download/v8.2.0/rtdetr-m.pt -O rtdetr-m.pt

In [ ]:
!rm -rf rf-detr
!git clone https://github.com/roboflow/rf-detr.git
import os
if os.path.exists('rf-detr'):
    %cd rf-detr
    !ls -R | grep ":$" | head -n 10
    if os.path.exists('requirements.txt'):
        !pip install -r requirements.txt
    if os.path.exists('setup.py') or os.path.exists('pyproject.toml'):
        !pip install .

### Next Steps
After you upload the video, we will need to run the detection script. Based on the repository structure, `IntegratedArchitecture.py` appears to be the main entry point. We might need to pass the video path as an argument.

In [ ]:
with open('requirements.txt', 'r') as f:
    lines = f.readlines()

new_lines = []
built_in_modules = ['json', 'os', 'time', 'sys', 'copy', 'math', 'datetime', 'pytorch', 'ultralytics']
for line in lines:
    is_built_in = False
    for module in built_in_modules:
        if module in line.strip().lower():
            is_built_in = True
            break
    if not is_built_in:
        new_lines.append(line)

with open('requirements.txt', 'w') as f:
    f.writelines(new_lines)

print('Modified requirements.txt:')
!cat requirements.txt

!pip install -r requirements.txt